# Module 1 · Lesson 08: LLMs Talking to Each Other

What happens when you connect **two or more LLMs** in a conversation?
This notebook explores multi-agent conversations using persona-based system prompts.

## What you will learn
1. How **conversation history** works with chat completions
2. Building a **round-robin agent loop**
3. Using **personas** to create distinct agent behaviors
4. The difference between multi-message vs single-prompt approaches

In [1]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv(Path.cwd().parent / ".env")

from openai import OpenAI
client = OpenAI()

def chat(messages, temperature=0.8, max_tokens=200):
    r = client.chat.completions.create(
        model="gpt-4o-mini", 
        messages=messages,
        temperature=temperature, 
        max_tokens=max_tokens
    )
    return r.choices[0].message.content

print("Ready")

Ready


---
## 1. Defining Personas

Each agent gets a distinct personality through its system prompt:

In [2]:
# ── Define 3 distinct personas ─────────────────────

PERSONAS = {
    "Optimist": {
        "system": "You are an enthusiastic optimist. Always see the bright side. "
                  "Keep responses under 3 sentences. Be specific, not generic.",
        "emoji": "\U0001F31E"
    },
    "Skeptic": {
        "system": "You are a thoughtful skeptic. Question assumptions and point out risks. "
                  "Keep responses under 3 sentences. Be constructive, not negative.",
        "emoji": "\U0001F9D0"
    },
    "Mediator": {
        "system": "You are a balanced mediator. Acknowledge both sides and find common ground. "
                  "Keep responses under 3 sentences. Synthesize, don't just repeat.",
        "emoji": "\u2696\uFE0F"
    }
}

print("Personas defined:")
for name, info in PERSONAS.items():
    print(f"  {info['emoji']} {name}: {info['system'][:60]}...")

Personas defined:
  🌞 Optimist: You are an enthusiastic optimist. Always see the bright side...
  🧐 Skeptic: You are a thoughtful skeptic. Question assumptions and point...
  ⚖️ Mediator: You are a balanced mediator. Acknowledge both sides and find...


---
## 2. The Conversation Loop

Each agent takes a turn, building on the conversation history:

In [ ]:
# ── Multi-agent debate ────────────────────────────

topic = "Should developers rely on AI coding assistants?"
num_rounds = 2
agent_names = list(PERSONAS.keys()) # ['Optimist', 'Skeptic', 'Mediator']

# Full conversation log (shared context)
conversation_log = []

print(f"Topic: {topic}")
print("=" * 60)

for round_num in range(num_rounds):
    print(f"\n--- Round {round_num + 1} ---")
    for name in agent_names:
        persona = PERSONAS[name]

        # Build messages for this agent
        messages = [{"role": "system", "content": persona["system"]}]

        if not conversation_log:
            # First speaker: introduce the topic
            messages.append({"role": "user", "content": f"Share your view on: {topic}"})
        else:
            # Subsequent speakers: show conversation so far
            context = "\n".join(f"{e['name']}: {e['text']}" for e in conversation_log)
            messages.append({"role": "user",
                           "content": f"Topic: {topic}\n\nConversation so far:\n{context}\n\n"
                                      f"Now share your perspective as {name}."})

        response = chat(messages)
        conversation_log.append({"name": name, "text": response})

        print(f"\n{persona['emoji']} {name}:")
        print(f"  {response}")

print(f"\nTotal messages: {len(conversation_log)}")

['Optimist', 'Skeptic', 'Mediator']
Topic: Should developers rely on AI coding assistants?

--- Round 1 ---

🌞 Optimist:
  Absolutely! AI coding assistants can boost productivity and enhance creativity by handling repetitive tasks, allowing developers to focus on more complex and innovative aspects of their projects. Plus, they can help catch errors and provide suggestions, making coding an even more enjoyable experience!

🧐 Skeptic:
  While AI coding assistants can indeed improve productivity, relying too heavily on them may lead to a decline in fundamental coding skills and a lack of deep understanding of the code being produced. Additionally, there are risks of misinformation or incorrect suggestions that could introduce bugs or security vulnerabilities if developers don’t critically evaluate the AI's output. It's essential to strike a balance between leveraging AI tools and maintaining robust coding practices.

⚖️ Mediator:
  Both perspectives recognize the potential benefits of AI

---
## 3. Conversation as Context

An alternative approach: send the **entire conversation** as a single prompt,
instead of per-agent message lists. This is often more reliable for 3+ agents:

In [ ]:
# ── Alternative: single-prompt approach ─────────────

summary_prompt = f"""Here's a debate between three AI personas on "{topic}":

{chr(10).join(f"{e['name']}: {e['text']}" for e in conversation_log)}

As a neutral observer, summarize:
1. The key point from each perspective
2. Where they agree
3. The strongest argument overall"""

summary = chat([{"role": "user", "content": summary_prompt}], max_tokens=400)
display(Markdown(f"### Debate Summary\n\n{summary}"))

---
## Key Takeaways

| Concept | Detail |
|---------|--------|
| **Personas** | System prompts create distinct agent behaviors |
| **Context building** | Each agent sees the full conversation history |
| **Round-robin** | Agents take turns, building on previous messages |
| **Single-prompt alt** | Pass entire convo as text for simpler multi-agent setups |

> **Exercise:** Change the topic and personas. Try 3 rounds. 
> What happens when you add a 4th persona (e.g., "Devil's Advocate")?

---
**Next:** `module_02_prompt_engineering` — Master the art of crafting effective prompts